In [ ]:
import ee

# ---------------------------------------------------------------------
# Authenticate and initialize Earth Engine
# ---------------------------------------------------------------------
# Run ee.Authenticate() the first time, or when credentials expire.
ee.Authenticate()

ee.Initialize(project="csumb-et-tools")  # or simply ee.Initialize()


## Extract land use fractions to HUC8 list

In [ ]:

import pandas as pd
url = "https://raw.githubusercontent.com/watrs-csumb/huc8basin-water-balance/main/data/metadata/Selected_basins.csv"
all_huc8s_df = pd.read_csv(url)
# ---------------------------------------------------------------------
# User inputs
# ---------------------------------------------------------------------
HUC8_LIST = all_huc8s_df.HUCID.astype(str).to_list()

CDL_YEAR = 2023

DRIVE_FOLDER = "GEE_exports"
EXPORT_DESCRIPTION = f"huc8_cdl_landuse_fractions_{CDL_YEAR}"
EXPORT_FILE_PREFIX = f"huc8_cdl_landuse_fractions_{CDL_YEAR}"


# ---------------------------------------------------------------------
# CDL class groupings
# ---------------------------------------------------------------------
# These are editable. CDL classes vary by year and include crop-specific
# classes plus NLCD-derived non-ag classes.
#
# Agriculture:
# Broad crop classes, orchards/vineyards, double-crop classes, pasture/hay.
# This intentionally excludes developed, water, wetlands, forest, shrubland.
AG_CLASSES = [
    1, 2, 3, 4, 5, 6,
    10, 11, 12, 13, 14,
    21, 22, 23, 24, 25, 26, 27, 28, 29,
    30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
    41, 42, 43, 44, 45, 46, 47, 48, 49,
    50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60,
    61,        # Fallow/Idle Cropland
    66, 67, 68, 69,
    70, 71, 72, 74, 75, 76, 77,
    204, 205, 206, 207, 208, 209,
    210, 211, 212, 213, 214, 215, 216,
    217, 218, 219,
    220, 221, 222, 223, 224, 225, 226,
    227, 228, 229, 230, 231, 232, 233,
    234, 235, 236, 237, 238, 239, 240,
    241, 242, 243, 244, 245, 246, 247,
    248, 249, 250, 254
]

# Forest:
# Includes older CDL forest code and current NLCD-derived forest classes.
FOREST_CLASSES = [
    63,        # Forest, older CDL non-ag code
    141, 142, 143
]

# Grassland:
# Includes older pasture/grass and current grassland/pasture classes.
# Note: 176 is usually "Grassland/Pasture" in newer CDL.
GRASSLAND_CLASSES = [
    62,
    176
]


# ---------------------------------------------------------------------
# Load HUC8 and CDL
# ---------------------------------------------------------------------
huc8_fc = (
    ee.FeatureCollection("USGS/WBD/2017/HUC08")
    .filter(ee.Filter.inList("huc8", HUC8_LIST))
)

cdl = (
    ee.ImageCollection("USDA/NASS/CDL")
    .filterDate(f"{CDL_YEAR}-01-01", f"{CDL_YEAR + 1}-01-01")
    .first()
    .select("cropland")
)


# ---------------------------------------------------------------------
# Helper: make binary masks and area images
# ---------------------------------------------------------------------
def class_mask(image, classes):
    """Return binary image: 1 where CDL class is in classes, else 0."""
    classes_ee = ee.List(classes)
    return image.remap(classes_ee, ee.List.repeat(1, classes_ee.length()), 0)


ag_mask = class_mask(cdl, AG_CLASSES)
forest_mask = class_mask(cdl, FOREST_CLASSES)
grass_mask = class_mask(cdl, GRASSLAND_CLASSES)

pixel_area = ee.Image.pixelArea()

# Area in square meters for each class group.
area_img = ee.Image.cat([
    pixel_area.updateMask(ag_mask).rename("ag_area_m2"),
    pixel_area.updateMask(forest_mask).rename("forest_area_m2"),
    pixel_area.updateMask(grass_mask).rename("grass_area_m2"),
    pixel_area.rename("total_area_m2")
])


# ---------------------------------------------------------------------
# Reduce over each HUC8
# ---------------------------------------------------------------------
def add_fractions(feature):
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=feature.geometry(),
        scale=30,
        maxPixels=1e13,
        tileScale=4
    )

    ag_area = ee.Number(stats.get("ag_area_m2", 0))
    forest_area = ee.Number(stats.get("forest_area_m2", 0))
    grass_area = ee.Number(stats.get("grass_area_m2", 0))
    total_area = ee.Number(stats.get("total_area_m2", 0))

    return ee.Feature(None, {
        "HUC8_id": feature.get("huc8"),
        "agland_fraction": ag_area.divide(total_area),
        "forrest_fraction": forest_area.divide(total_area),  # kept spelling as requested
        "grassland_fraction": grass_area.divide(total_area)
    })


results = huc8_fc.map(add_fractions)


# ---------------------------------------------------------------------
# Export to Google Drive
# ---------------------------------------------------------------------
task = ee.batch.Export.table.toDrive(
    collection=results,
    description=EXPORT_DESCRIPTION,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_FILE_PREFIX,
    fileFormat="CSV",
    selectors=[
        "HUC8_id",
        "agland_fraction",
        "forrest_fraction",
        "grassland_fraction"
    ]
)

task.start()

print("Export started.")
print("Task ID:", task.id)
print("Check task status in the Earth Engine Tasks tab or with task.status().")

Export started.
Task ID: 4GS4TPIOO7KDGV7TQTO3WN62
Check task status in the Earth Engine Tasks tab or with task.status().


## Extract elevation statistics to HUC8_List

In [ ]:
# ---------------------------------------------------------------------
# User inputs
# ---------------------------------------------------------------------
HUC8_LIST = all_huc8s_df.HUCID.astype(str).to_list()

DRIVE_FOLDER = "GEE_exports"
EXPORT_DESCRIPTION = "huc8_elevation_stats"
EXPORT_FILE_PREFIX = "huc8_elevation_stats"


# ---------------------------------------------------------------------
# Load HUC8 watersheds and subset to the requested HUC8s
# ---------------------------------------------------------------------
huc8_fc = (
    ee.FeatureCollection("USGS/WBD/2017/HUC08")
    .filter(ee.Filter.inList("huc8", HUC8_LIST))
)


# ---------------------------------------------------------------------
# Load elevation dataset
# ---------------------------------------------------------------------
# SRTM elevation is in meters.
elevation = ee.Image("USGS/SRTMGL1_003").select("elevation")


# ---------------------------------------------------------------------
# Calculate mean and standard deviation of elevation for each HUC8
# ---------------------------------------------------------------------
def add_elevation_stats(feature):
    stats = elevation.reduceRegion(
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ),
        geometry=feature.geometry(),
        scale=30,
        maxPixels=1e13,
        tileScale=4
    )

    return ee.Feature(None, {
        "HUC8_id": feature.get("huc8"),
        "elev_mean": stats.get("elevation_mean"),
        "elev_std": stats.get("elevation_stdDev")
    })


results = huc8_fc.map(add_elevation_stats)


# ---------------------------------------------------------------------
# Export table to Google Drive
# ---------------------------------------------------------------------
task = ee.batch.Export.table.toDrive(
    collection=results,
    description=EXPORT_DESCRIPTION,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_FILE_PREFIX,
    fileFormat="CSV",
    selectors=[
        "HUC8_id",
        "elev_mean",
        "elev_std"
    ]
)

task.start()

print("Export started.")
print("Task ID:", task.id)
print("Check task status in the Earth Engine Tasks tab or with task.status().")

Export started.
Task ID: Z7SDFRWYWD2RTKX2ZEEKO3IF
Check task status in the Earth Engine Tasks tab or with task.status().


## Extract Mean NDVI, Summer NDVI / Mean NDVI to HUC8 list